In [ ]:
import sqlite3

In [ ]:
import pandas as pd

In [ ]:
connect=sqlite3.connect(":memory:")

In [ ]:
schema_sql = '''
CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    track TEXT,
    math_score REAL
);

CREATE TABLE tracks (
    track_id INTEGER PRIMARY KEY,
    track_name TEXT UNIQUE NOT NULL,
    track_lead TEXT,
    difficulty TEXT
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    track_id INTEGER,
    FOREIGN KEY (track_id) REFERENCES tracks(track_id)
);

CREATE TABLE enrolments (
    enrolment_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    course_id INTEGER,
    final_grade REAL,
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);
'''

connect.executescript(schema_sql)
connect.commit()

print("Tables created successfully.")

Tables created successfully.


In [ ]:
students_data = [
    (1, "Ada", "Dataville", 91),
    (2, "Ben", "Dataville", 84),
    (3, "Chioma", "Mindspire", 78),
    (4, "David", "Mindspire", 88),
    (5, "Ella", "Webweave", 72),
    (6, "Femi", "Webweave", 65),
    (7, "Grace", "Questopia", 95),
    (8, "Hassan", "Questopia", 81),
    (9, "Ivy", "Appforge", 74),
    (10, "John", "Appforge", 69)
]

tracks_data = [
    (1, "Dataville", "Dr Osei", "Intermediate"),
    (2, "Mindspire", "Yemi Adeyemi", "Advanced"),
    (3, "Webweave", "Amaka Okon", "Intermediate"),
    (4, "Appforge", "Daniel James", "Advanced"),
    (5, "Questopia", "Sarah Bello", "Beginner")
]

courses_data = [
    (101, "Python Basics", 1),
    (102, "Data Analysis", 1),
    (103, "AI Explorers", 2),
    (104, "Web Design", 3),
    (105, "App Lab", 4),
    (106, "Game Design", 5)
]

enrolments_data = [
    (1, 1, 101, 92),
    (2, 2, 102, 85),
    (3, 3, 103, 79),
    (4, 4, 103, 90),
    (5, 5, 104, 73),
    (6, 7, 106, 96),
    (7, 8, 106, 82),
    (8, 9, 105, 76)
    # Femi and John intentionally have no enrolment.
]

connect.executemany("INSERT INTO students VALUES (?, ?, ?, ?)", students_data)
connect.executemany("INSERT INTO tracks VALUES (?, ?, ?, ?)", tracks_data)
connect.executemany("INSERT INTO courses VALUES (?, ?, ?)", courses_data)
connect.executemany("INSERT INTO enrolments VALUES (?, ?, ?, ?)", enrolments_data)
connect.commit()

print("Sample data inserted successfully.")

Sample data inserted successfully.


In [ ]:
def run_sql(sql_query):
    return pd.read_sql_query(sql_query, connect)

# SOLUTION — Practice 1

practice_1 = '''
SELECT
    s.name AS student_name,
    c.course_name,
    t.track_name
FROM students AS s
JOIN enrolments AS e
    ON s.student_id = e.student_id
JOIN courses AS c
    ON e.course_id = c.course_id
JOIN tracks AS t
    ON c.track_id = t.track_id
WHERE t.track_name = 'Dataville'
ORDER BY s.name;
'''

run_sql(practice_1)

,student_name,course_name,track_name
0,Ada,Python Basics,Dataville
1,Ben,Data Analysis,Dataville


In [ ]:
# SOLUTION — Practice 2

practice_2 = '''
SELECT
    a.name AS student_a,
    b.name AS student_b,
    a.track,
    a.math_score AS score_a,
    b.math_score AS score_b,
    ABS(a.math_score - b.math_score) AS difference
FROM students AS a
JOIN students AS b
    ON a.track = b.track
   AND a.student_id < b.student_id
WHERE ABS(a.math_score - b.math_score) <= 10
ORDER BY a.track, difference;
'''

run_sql(practice_2)

,student_a,student_b,track,score_a,score_b,difference
0,Ivy,John,Appforge,74.0,69.0,5.0
1,Ada,Ben,Dataville,91.0,84.0,7.0
2,Chioma,David,Mindspire,78.0,88.0,10.0
3,Ella,Femi,Webweave,72.0,65.0,7.0
